In [12]:
%pip install requests-html selenium lxml-html-clean

Note: you may need to restart the kernel to use updated packages.


In [14]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd

url = "https://sportmaniacs.com/es/races/maraton-de-santiago-2026/69ee59b0-6fe0-4c72-9bc7-487fac1f2bc4/rankings#/ranking/9007b928-176a-46db-9455-1e0c6b9a92c6/42K/ranking"

# Configurar opciones del navegador Chrome
options = webdriver.ChromeOptions()
options.add_argument('--headless')  # Ejecutar sin interfaz gráfica
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Iniciar el navegador
driver = webdriver.Chrome(options=options)

try:
    print("Accediendo a la página...")
    driver.get(url)
    
    # Esperar a que se cargue la tabla
    print("Esperando a que cargue la tabla...")
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.TAG_NAME, "tr"))
    )
    
    # Esperar un poco más para que cargue todo
    time.sleep(3)
    
    # Obtener el HTML renderizado
    html = driver.page_source
    
    # Parsear con BeautifulSoup
    soup = BeautifulSoup(html, 'html.parser')
    
    # Buscar la tabla de rankings
    table = soup.find('table')
    if table:
        print("✓ Tabla encontrada!")
        rows = table.find_all('tr')
        print(f"✓ Total de filas: {len(rows)}")
        
        # Extraer datos
        participants_data = []
        
        # Saltar encabezado (primera fila)
        for i, row in enumerate(rows[1:11]):  # Primeras 10 filas (excluyendo encabezado)
            cols = row.find_all(['td', 'th'])
            if len(cols) > 0:
                # Buscar nombre del participante y tiempo
                row_text = [col.get_text(strip=True) for col in cols]
                participants_data.append(row_text)
                print(f"{i+1}. {row_text}")
    else:
        print("✗ Tabla no encontrada")
        print(f"HTML length: {len(html)}")
        
finally:
    driver.quit()
    print("\nNavegador cerrado")

Accediendo a la página...
Esperando a que cargue la tabla...
✓ Tabla encontrada!
✓ Total de filas: 25
1. ['Michael Kirui12KenyaOverall Position2Gender2MalePremioNacionalCategory230 - 34 años MasculinoTime02:10:15']
2. ['HCHugo Catrileo Tapia3ChileOverall Position3Gender3MalePremioNacional1MasculinoCategory120 - 29 años MasculinoTime02:10:22']
3. ['Yitayal Atnafu Zerihun19EtiopíaOverall Position4Gender4MalePremioNacionalCategory330 - 34 años MasculinoTime02:12:12']
4. ['Daniel Cortes  Cortes2ChileOverall Position5Gender5MalePremioNacional2MasculinoCategory430 - 34 años MasculinoTime02:19:10']
5. ['Michael Pinela Pinela6Overall Position6Gender6MalePremioNacional3MasculinoCategory135 - 39 años MasculinoTime02:19:45']
6. ['Sebastian Traslaviña  Duran16Overall Position7Gender7MalePremioNacional4MasculinoCategory235 - 39 años MasculinoTime02:22:29']
7. ['Tigst Getnet Belew20EtiopíaOverall Position8Gender1FemalePremioNacionalCategory120 - 29 años FemeninoTime02:27:58']
8. ['Javier Martin  Tap

In [ ]:
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

url = "https://sportmaniacs.com/es/races/maraton-de-santiago-2026/69ee59b0-6fe0-4c72-9bc7-487fac1f2bc4/rankings#/ranking/9007b928-176a-46db-9455-1e0c6b9a92c6/42K/ranking"

# Configurar opciones del navegador
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=options)

try:
    print("🔄 Accediendo a la página...")
    driver.get(url)
    
    # Esperar a que cargue la tabla
    print("⏳ Esperando a que cargue la tabla...")
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.TAG_NAME, "tr"))
    )
    time.sleep(3)
    
    # Obtener el HTML renderizado
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    
    # Buscar la tabla
    table = soup.find('table')
    rows = table.find_all('tr')[1:101]  # Primeras 10 filas
    
    # Extraer datos de forma estructurada
    participants = []
    
    for idx, row in enumerate(rows, 1):
        cols = row.find_all(['td', 'th'])
        if len(cols) > 0:
            # Obtener el texto completo de la fila
            full_text = ' '.join([col.get_text(strip=True) for col in cols])
            
            # Extraer tiempo (formato HH:MM:SS)
            time_match = re.search(r'Time(\d{2}):(\d{2}):(\d{2})', full_text)
            if time_match:
                time_value = f"{time_match.group(1)}:{time_match.group(2)}:{time_match.group(3)}"
            else:
                time_value = "N/A"
            
            # Extraer nombre (está al principio, antes de números)
            # Buscar patrón: nombre + números/país
            name_match = re.match(r'^([A-Z][a-záéíóúñ\s]+?)(?:\d+|Overall)', full_text)
            if name_match:
                name = name_match.group(1).strip()
            else:
                # Alternativa: tomar el primer segmento
                name = full_text.split('Overall')[0].split('Gender')[0].strip()
                # Limpiar números del inicio
                name = re.sub(r'^[A-Z]{1,2}', '', name).strip()
            
            participants.append({
                'Posición': idx,
                'Participante': name,
                'Tiempo': time_value
            })
    
    # Crear DataFrame
    df_marathon = pd.DataFrame(participants)
    
    print("\n📊 Primeros 10 Participantes - Maratón de Santiago 2026 (42K):")
    print("=" * 70)
    print(df_marathon.to_string(index=False))
    print("=" * 70)
    
    # Guardar a CSV
    df_marathon.to_csv('marathon_results_top10.csv', index=False, encoding='utf-8')
    print(f"\n✓ Datos guardados en: marathon_results_top10.csv")
    
finally:
    driver.quit()

🔄 Accediendo a la página...
⏳ Esperando a que cargue la tabla...

📊 Primeros 10 Participantes - Maratón de Santiago 2026 (42K):
 Posición                   Participante   Tiempo
        1            ichael Kirui12Kenya 02:10:15
        2      Hugo Catrileo Tapia3Chile 02:10:22
        3 itayal Atnafu Zerihun19Etiopía 02:12:12
        4     aniel Cortes  Cortes2Chile 02:19:10
        5          ichael Pinela Pinela6 02:19:45
        6   ebastian Traslaviña  Duran16 02:22:29
        7     igst Getnet Belew20Etiopía 02:27:58
        8        avier Martin  Tapia4131 02:28:30
        9      icolas Paredes Cartes1521 02:29:37
       10   Florencia Borelli44Argentina 02:29:45

✓ Datos guardados en: marathon_results_top10.csv


In [16]:
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

url = "https://sportmaniacs.com/es/races/maraton-de-santiago-2026/69ee59b0-6fe0-4c72-9bc7-487fac1f2bc4/rankings#/ranking/9007b928-176a-46db-9455-1e0c6b9a92c6/42K/ranking"

# Configurar opciones del navegador
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=options)

try:
    print("🔄 Accediendo a la página...")
    driver.get(url)
    
    # Esperar a que cargue la tabla
    print("⏳ Esperando a que cargue la tabla...")
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.TAG_NAME, "tr"))
    )
    time.sleep(3)
    
    # Obtener el HTML renderizado
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    
    # Buscar la tabla
    table = soup.find('table')
    rows = table.find_all('tr')[1:11]  # Primeras 10 filas
    
    # Extraer datos de forma estructurada
    participants = []
    
    for idx, row in enumerate(rows, 1):
        cols = row.find_all(['td', 'th'])
        if len(cols) > 0:
            # Procesar cada celda
            cell_data = []
            for col in cols:
                text = col.get_text(strip=True)
                if text:
                    cell_data.append(text)
            
            # Unir los datos
            full_text = ' '.join(cell_data)
            
            # Extraer tiempo (formato HH:MM:SS)
            time_match = re.search(r'(\d{2}):(\d{2}):(\d{2})', full_text)
            if time_match:
                time_value = time_match.group(0)
            else:
                time_value = "N/A"
            
            # Extraer nombre: está en la primera celda antes de números
            # El patrón es: Nombre + Números + País/Región
            first_cell = cell_data[0] if cell_data else ""
            
            # Limpiar: extraer solo las letras y espacios del inicio
            name = re.sub(r'(\d+.*)', '', first_cell).strip()
            # Si quedó vacío, usar la primera celda completa
            if not name or len(name) < 3:
                name = first_cell
            
            participants.append({
                'Posición': idx,
                'Participante': name,
                'Tiempo': time_value
            })
    
    # Crear DataFrame
    df_marathon = pd.DataFrame(participants)
    
    print("\n📊 TOP 10 PARTICIPANTES - Maratón de Santiago 2026 (42K):")
    print("=" * 70)
    print(df_marathon.to_string(index=False))
    print("=" * 70)
    
    # Guardar a CSV
    df_marathon.to_csv('marathon_results_top10.csv', index=False, encoding='utf-8')
    print(f"\n✓ Datos guardados en: marathon_results_top10.csv")
    print(f"\nDataFrame disponible como: df_marathon")
    
finally:
    driver.quit()

🔄 Accediendo a la página...
⏳ Esperando a que cargue la tabla...

📊 TOP 10 PARTICIPANTES - Maratón de Santiago 2026 (42K):
 Posición                Participante   Tiempo
        1               Michael Kirui 02:10:15
        2       HCHugo Catrileo Tapia 02:10:22
        3      Yitayal Atnafu Zerihun 02:12:12
        4       Daniel Cortes  Cortes 02:19:10
        5       Michael Pinela Pinela 02:19:45
        6 Sebastian Traslaviña  Duran 02:22:29
        7          Tigst Getnet Belew 02:27:58
        8        Javier Martin  Tapia 02:28:30
        9      Nicolas Paredes Cartes 02:29:37
       10         FBFlorencia Borelli 02:29:45

✓ Datos guardados en: marathon_results_top10.csv

DataFrame disponible como: df_marathon


In [18]:
df_marathon.head()


,Posición,Participante,Tiempo
0,1,Michael Kirui,02:10:15
1,2,Hugo Catrileo Tapia,02:10:22
2,3,Yitayal Atnafu Zerihun,02:12:12
3,4,Daniel Cortes Cortes,02:19:10
4,5,Michael Pinela Pinela,02:19:45


In [17]:
# Limpiar los nombres removiendo códigos de categoría al inicio
def clean_name(name):
    """Limpiar nombres removiendo códigos como HC, FB, etc."""
    # Remover códigos de 2 letras al inicio si van seguidos de mayúscula
    name = re.sub(r'^[A-Z]{2}(?=[A-Z])', '', name)
    return name.strip()

# Aplicar limpieza
df_marathon['Participante'] = df_marathon['Participante'].apply(clean_name)

print("\n📊 TOP 10 PARTICIPANTES (LIMPIO) - Maratón de Santiago 2026 (42K):")
print("=" * 75)
print(df_marathon.to_string(index=False))
print("=" * 75)

# Guardar versión limpia
df_marathon.to_csv('marathon_results_top10.csv', index=False, encoding='utf-8')
print(f"\n✓ Datos limpios guardados en: marathon_results_top10.csv")
print(f"\n📋 Resumen:")
print(f"  - Primer lugar: {df_marathon.iloc[0]['Participante']} - {df_marathon.iloc[0]['Tiempo']}")
print(f"  - Mejor tiempo femenino: {df_marathon[df_marathon['Participante'].str.contains('Florencia|Tigst', case=False)].iloc[0]['Participante']} - {df_marathon[df_marathon['Participante'].str.contains('Florencia|Tigst', case=False)].iloc[0]['Tiempo']}")


📊 TOP 10 PARTICIPANTES (LIMPIO) - Maratón de Santiago 2026 (42K):
 Posición                Participante   Tiempo
        1               Michael Kirui 02:10:15
        2         Hugo Catrileo Tapia 02:10:22
        3      Yitayal Atnafu Zerihun 02:12:12
        4       Daniel Cortes  Cortes 02:19:10
        5       Michael Pinela Pinela 02:19:45
        6 Sebastian Traslaviña  Duran 02:22:29
        7          Tigst Getnet Belew 02:27:58
        8        Javier Martin  Tapia 02:28:30
        9      Nicolas Paredes Cartes 02:29:37
       10           Florencia Borelli 02:29:45

✓ Datos limpios guardados en: marathon_results_top10.csv

📋 Resumen:
  - Primer lugar: Michael Kirui - 02:10:15
  - Mejor tiempo femenino: Tigst Getnet Belew - 02:27:58
